# AIM:
create evaluation workflow. Taking the manually extracted Ci impacts (validation set) and compare it with the CI impacts (llm_geolocations.ipynb) extrracted by the first LLM 1. 
As a first step the evaluation should be done only for the direct CI impacts - CI type, damage and geolocation

Issue:
* What is needed an approach that recognizes when an direct impact case is not detected by the model
Idea: 
* Split the original texts passed to the model on the exact chunks as again
* Then chunkwise check if the CI impacts from the validation set correspond in number and their textual similarity to the CI impacts infered by the LLM 1 and Entity Linking 

## Semantic Textual Similarity (STS)

Calculating the STS for both model configurations (chain of prompts, orchestration of models)
The outputs are cosine similarity scores for similar model outputs per chunk. They are ranked by score for each model, restricted to the top 20 results.  


In [14]:
import os
import sys
import io
import numpy as np

from pathlib import Path
import pickle
import time
import warnings
import subprocess
import importlib

import spacy
import pandas as pd
import torch
import pyarrow as pa
import pyarrow.parquet as pq


# from utils.training import topic_search, topic_search_lm

sys.path.append('../')
from src.settings import settings as s
import src.document_cleaning as dc

### Direct CI impacts: LLM 1 vs domain-expertise 

In [15]:
#  Suppress future warnings from PyTorch
warnings.filterwarnings("ignore", category=FutureWarning)


#  Define data dir where tags.csv and domain-expertise derived tag lists are found 
VALID_DATA_FILENAME = "table_ci_impacts_sm.csv"
PATH_VALID_DATA = s.PATH_VALID_DATA
PATH_EVAL_RESULT = s.PATH_EVAL_RESULT
LLM_DATA_FILEPATH = Path(s.PATH_LLM_DATA / "llm1_2026-02-11_7of9_old.csv") #s.LLM_DATA_FILENAME)
SIMILARITY_LLM_FILENAME = s.SIMILARITY_LLM_FILENAME

df_valid = pd.read_csv(
    PATH_VALID_DATA / VALID_DATA_FILENAME,
    usecols=["publication_id", "ci1_type", "ci1_damage", "ci1_location"]
)
print(len(df_valid))
## pre-process: 
# remove undone entries
df_valid = df_valid[~df_valid[
        ["publication_id", "ci1_type", "ci1_damage", "ci1_location"]
    ].astype(str).apply(lambda x: x.str.contains("xx")).any(axis=1)]

print(len(df_valid))



## prediction data
df_pred = pd.read_csv(
    LLM_DATA_FILEPATH,
    usecols=["citation_id", "chunk_id", "infrastructure_type", "damage", "location"]
)
## citation alignment
print(df_pred["citation_id"])
# df_pred["citation_id"] = df_pred["citation_id"].map(dc.extract_citation_info) # FIXME as used with new funct returning author, year, title
# df_pred["citation_id"] = df_pred["citation_id"].apply(dc.extract_citation_info)
# print(df_pred["citation_id"])


132
123
0      Karakatsani 2023
1      Karakatsani 2023
2      Karakatsani 2023
3      Karakatsani 2023
4      Karakatsani 2023
             ...       
351      Wildhagen 2013
352      Wildhagen 2013
353      Wildhagen 2013
354      Wildhagen 2013
355      Wildhagen 2013
Name: citation_id, Length: 356, dtype: object


#### Create vectors for evaluation and LLm output

In [16]:
# !uv run python -m spacy download en_core_web_lg

# os.chdir("/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval")
nlp = spacy.load("en_core_web_lg")

In [17]:
## load english model with word vectors included
s.SPACY_MODEL = "en_core_web_lg"

print(f"Try loading spaCy language model ({s.SPACY_MODEL}) for remote instance (e.g., cluster)")
try: 
    nlp = spacy.load(s.SPACY_MODEL)
except (OSError, ValueError):
    print(f"spaCy language model '{s.SPACY_MODEL}' not found. Downloading ...")
    subprocess.check_call(["uv", "pip", "install", "spacy-transformers"])
    subprocess.check_call(["uv", "run", "python", "-m", "spacy", "download", s.SPACY_MODEL])
    nlp = spacy.load(s.SPACY_MODEL)
        

# !uv run python -m spacy download en_core_web_lg
# nlp = spacy.load("en_core_web_lg")


Try loading spaCy language model (en_core_web_lg) for remote instance (e.g., cluster)


In [18]:
def cosine_similarity(vector_a, vector_b):
    dot_product = np.dot(vector_a, vector_b)
    magnitude_a = np.linalg.norm(vector_a)
    magnitude_b = np.linalg.norm(vector_b)
    return dot_product / (magnitude_a * magnitude_b)




In [19]:
# ## unify citation column

# # get corresponding document from df_vald
# citation_pattern = r"(.*?)(\d{4})(.*)" # split at first occurrence of year
# # df_pred["publication_id"] = df_pred["citation"].map(dc.extract_citation_info)
# df_pred["citation"].map(dc.extract_citation_info)
# # df_pred = df_pred.rename({"citation": "publiation_id"})
# # df_pred.drop("citation", inplace=True)
# df_pred
# # try:
# #     authors, year, _ = re.findall(citation_pattern, filename)[0]
# #     citation = f"{authors} {year}"



#### for each validation entry, search for all prediction cases of the same document 

In [20]:
## get same impact entries
columns_valid = ["ci1_type", "ci1_damage", "ci1_location"]
columns_pred = ["infrastructure_type", "damage", "location"]



for column_valid, column_pred in zip(columns_valid, columns_pred):

    print(f" --------- Processing column pair: {column_valid} - {column_pred} ------------")
    
    df_valid_pred_all = pd.DataFrame()
    citations_list = []

    ## for each validation record
    for i in range(len(df_valid)):
        
        highest_similarity_score = 0.00
        
        ## needed to traceback info when entry is missing in pred. DS
        # chunk_id_value_valid = df_valid.chunk_id[i]

        # select nth validation record
        df_valid_entry = df_valid.iloc[i]
        citation_str = df_valid_entry.publication_id
        citations_list.append(citation_str)


        # get all corresponding prediction records
        df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]

        #  handle on NANs
        df_pred_entries[column_pred] = np.where(df_pred_entries[column_pred].isna(), "nan", df_pred_entries[column_pred])
        # df_pred_entries[column_pred] = df_pred_entries[column_pred].astype(str)
        # remove double whitespaces
        # df_pred_doc[column_pred] = df_pred_doc[column_pred].replace("  ", " ")
        # df_valid_entries[column_valid] = df_valid_entries[column_valid].replace("  ", " ")

        # skip when validation entry has no value
        if df_valid_entry[column_valid] is np.nan:
            continue

        # vector of validiation entry 
        valid_impact = df_valid_entry[column_valid]
        valid_vec = nlp(valid_impact).vector

        # print(" ------- Searching for citation:", citation_str, " in predictions ------- ")

        # Compute similarity between each validation CI impact case and all potential predicted CI impact cases (cross-product)
        for j in range(len(df_pred_entries[column_pred])):

            if df_pred_entries[column_pred].iloc[j] == "nan":
                continue

            pred_impact = df_pred_entries[column_pred].iloc[j]

            pred_vec = nlp(pred_impact).vector
            similarity_score = cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
            # print(f"Similarity {i}-{j}: {similarity_score}")

            # print(f"Searching for highest similarity ... ")
            ## get only pair with highest similarity
            if similarity_score > highest_similarity_score:
                
                highest_similarity_score = similarity_score
                
                dict_pair = {
                    "impact_valid": valid_impact, 
                    "impact_pred": pred_impact, 
                    "similarity": highest_similarity_score,
                    "citation": citation_str,
                    "chunk_id_pred": (df_pred.chunk_id[i],  df_pred.chunk_id[j])
                }
            else:
                continue

        df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


    print(f" ---------- Evaluation summary statistics - {column_pred}: -----------")
    print(df_valid_pred_all.similarity.describe())



    SIMILARITY_FILENAME = f'{column_pred}_{SIMILARITY_LLM_FILENAME}'
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    print("Saving evaluation statistics and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
    with open(SIMILARITY_FILEPATH, 'w') as f:
        # results
        df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
        df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
        pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
        #   summary statistics
        df_valid_pred_all_stats = df_valid_pred_all.describe()
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_valid_pred_all_stats.to_json(f, indent=4)



    similarity_threshold = 0.75
    df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] >= similarity_threshold
    print(f"Number of similar impact cases (similarity >= {similarity_threshold}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}\n")

    df_valid_pred_all=  df_valid_pred_all[df_valid_pred_all['similarity'] >= similarity_threshold]

    SIMILARITY_FILENAME = f'{column_pred}_75_{SIMILARITY_LLM_FILENAME}'
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    with open(SIMILARITY_FILEPATH, 'w') as f:
        # results
        df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
        df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
        pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
        #   summary statistics
        df_valid_pred_all_stats = df_valid_pred_all.describe()
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_valid_pred_all_stats.to_json(f, indent=4)


 --------- Processing column pair: ci1_type - infrastructure_type ------------


/tmp/ipykernel_958095/2629576861.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pred_entries[column_pred] = np.where(df_pred_entries[column_pred].isna(), "nan", df_pred_entries[column_pred])
/tmp/ipykernel_958095/2640203756.py:5: RuntimeWarning: invalid value encountered in scalar divide
  return dot_product / (magnitude_a * magnitude_b)


 ---------- Evaluation summary statistics - infrastructure_type: -----------
count    106.000000
mean       0.776583
std        0.124543
min        0.480856
25%        0.673384
50%        0.774533
75%        0.839790
max        1.000000
Name: similarity, dtype: float64
Saving evaluation statistics and scores to  infrastructure_type_smlrty_llm1_2026-02-11 [.parquet, _stats.json]
Number of similar impact cases (similarity >= 0.75): 59 out of 106

 --------- Processing column pair: ci1_damage - damage ------------


/tmp/ipykernel_958095/2629576861.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pred_entries[column_pred] = np.where(df_pred_entries[column_pred].isna(), "nan", df_pred_entries[column_pred])


 ---------- Evaluation summary statistics - damage: -----------
count    94.000000
mean      0.744826
std       0.202854
min       0.496814
25%       0.496814
50%       0.773405
75%       0.906963
max       1.000000
Name: similarity, dtype: float64
Saving evaluation statistics and scores to  damage_smlrty_llm1_2026-02-11 [.parquet, _stats.json]
Number of similar impact cases (similarity >= 0.75): 57 out of 94

 --------- Processing column pair: ci1_location - location ------------


/tmp/ipykernel_958095/2629576861.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pred_entries[column_pred] = np.where(df_pred_entries[column_pred].isna(), "nan", df_pred_entries[column_pred])
/tmp/ipykernel_958095/2640203756.py:5: RuntimeWarning: invalid value encountered in scalar divide
  return dot_product / (magnitude_a * magnitude_b)


 ---------- Evaluation summary statistics - location: -----------
count    94.000000
mean      0.717121
std       0.191070
min       0.269494
25%       0.589181
50%       0.589181
75%       0.973928
max       1.000000
Name: similarity, dtype: float64
Saving evaluation statistics and scores to  location_smlrty_llm1_2026-02-11 [.parquet, _stats.json]
Number of similar impact cases (similarity >= 0.75): 36 out of 94



In [ ]:
# ## get same impact entries
# columns_valid = ["ci1_type", "ci1_damage", "ci1_location"]
# columns_pred = ["infrastructure_type", "damage", "location"]



# for column_valid, column_pred in zip(columns_valid, columns_pred):

#     print(f" --------- Processing column pair: {column_valid} - {column_pred} ------------")
    
#     df_valid_pred_all = pd.DataFrame()
#     citations_list = []

#     ## for each validation record
#     for i in range(len(df_valid)):
        
#         highest_similarity_score = 0.00
        
#         ## needed to traceback info when entry is missing in pred. DS
#         # chunk_id_value_valid = df_valid.chunk_id[i]

#         # select nth validation record
#         df_valid_entry = df_valid.iloc[i]
#         citation_str = df_valid_entry.publication_id
#         citations_list.append(citation_str)


#         # get all corresponding prediction records
#         df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]

#         #  handle on NANs
#         df_pred_entries[column_pred] = np.where(df_pred_entries[column_pred].isna(), "nan", df_pred_entries[column_pred])
#         # df_pred_entries[column_pred] = df_pred_entries[column_pred].astype(str)
#         # remove double whitespaces
#         # df_pred_doc[column_pred] = df_pred_doc[column_pred].replace("  ", " ")
#         # df_valid_entries[column_valid] = df_valid_entries[column_valid].replace("  ", " ")

#         # skip when validation entry has no value
#         if df_valid_entry[column_valid] is np.nan:
#             continue

#         # vector of validiation entry 
#         valid_impact = df_valid_entry[column_valid]
#         valid_vec = nlp(valid_impact).vector

#         # print(" ------- Searching for citation:", citation_str, " in predictions ------- ")

#         # Compute similarity between each validation CI impact case and all potential predicted CI impact cases (cross-product)
#         for j in range(len(df_pred_entries[column_pred])):

#             if df_pred_entries[column_pred].iloc[j] == "nan":
#                 continue

#             pred_impact = df_pred_entries[column_pred].iloc[j]

#             pred_vec = nlp(pred_impact).vector
#             similarity_score = cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
#             # print(f"Similarity {i}-{j}: {similarity_score}")

#             # print(f"Searching for highest similarity ... ")
#             ## get only pair with highest similarity
#             if similarity_score > highest_similarity_score:
                
#                 highest_similarity_score = similarity_score
                
#                 dict_pair = {
#                     "impact_valid": valid_impact, 
#                     "impact_pred": pred_impact, 
#                     "similarity": highest_similarity_score,
#                     "citation": citation_str,
#                     "chunk_id_pred": (df_pred.chunk_id[i],  df_pred.chunk_id[j])
#                 }
#             else:
#                 continue

#         df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


#     print(f" ---------- Evaluation summary statistics - {column_pred}: -----------")
#     print(df_valid_pred_all.similarity.describe())



#     SIMILARITY_FILENAME = f'{column_pred}_{SIMILARITY_LLM_FILENAME}'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     print("Saving evaluation statistics and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)



#     similarity_threshold = 0.75
#     df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] >= similarity_threshold
#     print(f"Number of similar impact cases (similarity >= {similarity_threshold}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}\n")

#     df_valid_pred_all=  df_valid_pred_all[df_valid_pred_all['similarity'] >= similarity_threshold]

#     SIMILARITY_FILENAME = f'{column_pred}_75_{SIMILARITY_LLM_FILENAME}'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)


PosixPath('../data/evaluation_results/smlrty_llm1_2026-02-11_7of9.csv.parquet_location_75.csv')

In [81]:
columns_pred

['infrastructure_type', 'damage', 'location']

True

### Load parquet file

In [22]:

columns_pred = ["infrastructure_type", "damage", "location"]

In [ ]:
column_pred = "infrastructure_type"
SIMILARITY_FILENAME = f'llm1_similarity_{column_pred}_75.parquet'
SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    df = pd.read_parquet(SIMILARITY_FILEPATH, engine='pyarrow')
    display(df)

## Archive

In [ ]:
# ## get same impact entries
# columns_valid = ["ci1_type", "ci1_damage", "ci1_location"]
# columns_pred = ["infrastructure_type", "damage", "location"]



## iterate over predictions andd search for eahc predciotn reocrds for corresponding vlaid cases 

# for column_valid, column_pred in zip(columns_valid, columns_pred):

#     print(f" --------- Processing column pair: {column_valid} - {column_pred} ------------")
    
#     df_valid_pred_all = pd.DataFrame()
#     citations_list = []

#     ## for each validation record
#     for i in range(len(df_valid)):
        
#         highest_similarity_score = 0.00
        
#         ## needed to traceback info when entry is missing in pred. DS
#         # chunk_id_value_valid = df_valid.chunk_id[i]

#         # select nth validation record
#         df_valid_entry = df_valid.iloc[i]
#         citation_str = df_valid_entry.publication_id
#         citations_list.append(citation_str)
#         print(" ------- Searching for citation:", citation_str, " in predictions ------- ")


#         # get all corresponding prediction records
#         df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]
#         #  handle on NANs
#         df_pred_entries[column_pred] = np.where(df_pred_entries[column_pred].isna(), "nan", df_pred_entries[column_pred])
#         # df_pred_entries[column_pred] = df_pred_entries[column_pred].astype(str)
#         # remove double whitespaces
#         # df_pred_doc[column_pred] = df_pred_doc[column_pred].replace("  ", " ")
#         # df_valid_entries[column_valid] = df_valid_entries[column_valid].replace("  ", " ")

#         # skip when validation entry ha no value
#         if df_valid_entry[column_valid] is np.nan:
#             continue

#         # vector of validiation entry 
#         valid_impact = df_valid_entry[column_valid]
#         valid_vec = nlp(valid_impact).vector


#         # Compute similarity between each predicted impact case and all potential validation impact cases (cross-product)
#         # print(f"Searching for highest similarity of`{pred_impact}` in validation set ... ")
#         for j in range(len(df_pred_entries[column_pred])):

#             if df_pred_entries[column_pred].iloc[j] == "nan":
#                 continue

#             pred_impact = df_pred_entries[column_pred].iloc[j]

#             pred_vec = nlp(pred_impact).vector
#             similarity_score = cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
#             # print(f"Similarity {i}-{j}: {similarity_score}")

#             ## get only pair with highest similarity
#             if similarity_score > highest_similarity_score:
                
#                 highest_similarity_score = similarity_score
                
#                 dict_pair = {
#                     "impact_valid": valid_impact, 
#                     "impact_pred": pred_impact, 
#                     "similarity": highest_similarity_score,
#                     "citation": citation_str,
#                     "chunk_id_pred": df_pred.chunk_id[i]
#                 }
#             else:
#                 continue

#         df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


#     print(" ---------- Evaluation summary statistics: -----------")
#     print(df_valid_pred_all.similarity.describe())



#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{column_pred}.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     print("Saving evaluation statistics and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)



#     similarity_threshold = 0.75
#     df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] >= similarity_threshold
#     print(f"Number of similar impact cases (similarity >= {similarity_threshold}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}")

#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{column_pred}_75.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:

# #  Define folder for handling and writing outputs
# def write_to_file(data, out_folder, filename):
#     """Convert output to DataFrame and write to file"""
#     df = pd.DataFrame(list(data), columns=['tag', 'sts_score'])
#     #  Sort the DataFrame by similarity (explicitly)
#     df = df.sort_values(by='sts_score', ascending=False)
#     #  Assign integers to ranking
#     df['rank'] = df['sts_score'].rank(method='first', ascending=False).astype(int)
#     #  Only keep the first 20 resulting tags
#     df = df.head(50)
#     #  Save to file
#     df.to_csv(out_folder / f'{filename}_output.csv', index=False)

# #  Fill run metrics to dictionary
# def handle_metrics(metrics, model_name, length, end_time, start_time):
#     print(f'-> Took {end_time - start_time:.2f} seconds. Number of tags: {length}.')
#     metrics.append({
#         'modelname': model_name,
#         'runtime': round(end_time - start_time, 2),
#         'tagcount': length
#     })
#     return metrics

# class CPU_Unpickler(pickle.Unpickler):
#     """Fix for having issues with loading models on CPU"""
#     def find_class(self, module, name):
#         if module == 'torch.storage' and name == '_load_from_bytes':
#             return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
#         else: return super().find_class(module, name)
